# 01 — Prepare data

Wealth ML SageMaker Foundation (WSF).

**Input:** `metadata_csv` describes the columns and gives reference values.
**Output:** `sample.csv` contains generated data rows; the feature mapping describes
how those columns belong to Feature Groups.

Both modes generate the same local sample first. `csv_direct` stops there;
`feature_store` additionally ingests the sample into Feature Store. The pipeline
queries the store later, when GetData runs. This notebook does not run GetData.

Fill the input cells below. Preset values are kept separately in **Defaults**.
Nothing silently substitutes the included example metadata.

## Your metadata file

- Paste the path to the table metadata CSV, containing column definitions and reference values.
- This is the input used to generate dummy data, not an already generated sample.csv.
- Provide this file for either mode; this cell contains no presets.

In [ ]:
metadata_csv = ""  # Table metadata CSV. Absolute path or path relative to this checkout.

## Feature Store connection — fill only for feature_store

- Leave both values empty when using CSV-direct mode.
- For Feature Store mode, supply the existing bucket and offline-writer role.
- These values are used for ingestion; the pipeline role is supplied separately in notebook 02.

In [ ]:
bucket = ""
role_arn = ""  # Feature Store offline-writer IAM role ARN.

## Defaults — change only when needed

- Keep csv_direct to generate the sample without calling AWS; select feature_store to ingest it too.
- Keep wsf-dev for this deployment, separate from the earlier source-dev setup.
- Adjust sample size, keys, event date and type overrides here.
- Missing Feature Groups are not created unless explicitly enabled.

In [ ]:
data_source = "csv_direct"  # Or "feature_store". Use the same mode in notebook 02.
namespace = "wsf-dev"
sample_rows = 100
record_id = "customer_id"
event_time = "feature_asof_dt"
sample_asof = "2026-01-01T00:00:00Z"
type_overrides = {}             # column_name: binary / integer / continuous / string.
athena_workgroup = "primary"
kms_key = ""
create_missing_groups = False  # Platform normally creates groups separately.
verify_offline_readback = False # Optional Athena readback; offline delivery can take time.

## Generate the sample

- Read your metadata and cycle through its reference values to create synthetic rows.
- Write `sample.csv`, the Feature Store mapping and source-column mapping locally.
- Print the exact sample path and preview its columns; CSV mode makes no AWS calls.

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
import importlib
import json
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-features/data.py").is_file())
sys.path[:0] = [str(ROOT / "mlops-features"), str(ROOT / "mlops-models/src")]
import mlops_features
assert Path(mlops_features.__file__).resolve().is_relative_to(ROOT.resolve()), (
    f"STOP: this kernel loaded code from {mlops_features.__file__}. Restart this notebook's kernel for the new checkout."
)
import mlops_features.metadata as metadata_module
generate_sample = importlib.reload(metadata_module).generate_sample
from mlops_features.feature_mapping import read_mapping, group_columns, group_name
from mlops_features.data import validate_delivery, ingest_csv, read_dataset

assert data_source in {"csv_direct", "feature_store"}, "Choose csv_direct or feature_store."
assert metadata_csv.strip(), "STOP: set metadata_csv. No example file is selected automatically."
metadata_path = Path(metadata_csv).expanduser()
if not metadata_path.is_absolute():
    metadata_path = ROOT / metadata_path
metadata_path = metadata_path.resolve()
assert metadata_path.is_file(), f"STOP: metadata CSV does not exist: {metadata_path}"
metadata_csv = str(metadata_path)
RUN = ROOT / ".runs/feature-store" / namespace / uuid4().hex
sample = generate_sample(
    metadata_csv, RUN, rows=sample_rows, record_id=record_id, event_time=event_time,
    asof=sample_asof, type_overrides=type_overrides, deduplicate=True,
)
mapping = read_mapping(sample["mapping"], record_id, event_time)
groups = {
    key: {"name": group_name(namespace, key), "columns": columns}
    for key, columns in group_columns(mapping, record_id, event_time).items()
}
validate_delivery(sample["sample"], {"record_id": record_id, "event_time": event_time, "mapping": mapping})
df = pd.read_csv(sample["sample"], dtype=str, keep_default_na=False)
print("Metadata input:", metadata_csv)
print("Local sample:", sample["sample"].resolve())
print("Mapping:", sample["mapping"].resolve())
print("Source column mapping:", sample["metadata"].resolve())
print("Rows:", sample["rows"], "| Features:", sample["features"], "| Source tables:", sample["tables"])
print("Feature groups:", {group["name"]: len(group["columns"]) + 1 for group in groups.values()})
print("Storage types are suggestions from the five reference values. Review before creating groups.")
display(pd.DataFrame(mapping).head(20))
display(df.iloc[:5, :10])

## Connect to Feature Store

- Skip this cell entirely in CSV mode.
- Show the AWS caller, offline-writer role and S3 destination.
- Prepare the binding that pipeline GetData will use to query delivered features.

In [ ]:
if data_source == "feature_store":
    import boto3
    from botocore.exceptions import ClientError
    import time

    bucket = bucket.strip().removeprefix("s3://").rstrip("/")
    assert bucket and "/" not in bucket and role_arn.strip(), "Fill in bucket and role_arn."
    session = boto3.Session()
    sm = session.client("sagemaker")
    caller = session.client("sts").get_caller_identity()
    settings = {
        "namespace": namespace, "account": caller["Account"], "region": session.region_name,
        "bucket": bucket, "prefix": namespace + "/features/", "record_id": record_id,
        "event_time": event_time, "mapping": mapping,
        "mapping_sha256": hashlib.sha256(sample["mapping"].read_bytes()).hexdigest(),
        "groups": groups, "workgroup": athena_workgroup, "kms_key": kms_key,
        "query_output": f"s3://{bucket}/{namespace}/features/queries/",
        "source_metadata": json.loads(sample["metadata"].read_text()),
    }
    print("Caller:", caller["Arn"])
    print("Offline writer role:", role_arn)
    print("Storage:", f"s3://{bucket}/{settings['prefix']}")
    print("Athena workgroup:", athena_workgroup)
    print("Local evidence:", RUN.resolve())
else:
    print("Skipped: CSV mode.")

## Check the Feature Groups

- Skip this cell entirely in CSV mode.
- Describe each expected group and check its schema and offline storage.
- Create missing groups only if explicitly enabled; otherwise stop with the required group name.

In [ ]:
if data_source == "feature_store":
    offline_uri = f"s3://{bucket}/{settings['prefix']}offline/"
    for group in groups.values():
        definitions = [{"FeatureName": name, "FeatureType": kind}
                       for name, kind in {**group["columns"], "delivery_id": "String"}.items()]
        storage = {"S3Uri": offline_uri}
        if kms_key:
            storage["KmsKeyId"] = kms_key
        try:
            description = sm.describe_feature_group(FeatureGroupName=group["name"])
        except ClientError as error:
            if error.response["Error"]["Code"] != "ResourceNotFound":
                raise
            if not create_missing_groups:
                raise RuntimeError(f"Feature Group {group['name']} is missing. Ask the platform to create it, or explicitly enable create_missing_groups with an authorized role.") from error
            response = sm.create_feature_group(
                FeatureGroupName=group["name"], RecordIdentifierFeatureName=record_id,
                EventTimeFeatureName=event_time, FeatureDefinitions=definitions,
                RoleArn=role_arn.strip(), OnlineStoreConfig={"EnableOnlineStore": False},
                OfflineStoreConfig={"S3StorageConfig": storage, "DisableGlueTableCreation": False},
            )
            print("Created:", response["FeatureGroupArn"])
            description = sm.describe_feature_group(FeatureGroupName=group["name"])
        deadline = time.monotonic() + 900
        while description["FeatureGroupStatus"] == "Creating" and time.monotonic() < deadline:
            print("Waiting for:", group["name"], flush=True)
            time.sleep(15)
            description = sm.describe_feature_group(FeatureGroupName=group["name"])
        assert description["FeatureGroupStatus"] == "Created", description
        actual_definitions = {item["FeatureName"]: item["FeatureType"] for item in description["FeatureDefinitions"]}
        assert actual_definitions == {item["FeatureName"]: item["FeatureType"] for item in definitions}, "Existing schema differs."
        assert description["RecordIdentifierFeatureName"] == record_id and description["EventTimeFeatureName"] == event_time
        assert description["RoleArn"] == role_arn.strip(), "Existing offline writer role differs."
        actual_storage = description["OfflineStoreConfig"]["S3StorageConfig"]
        assert actual_storage["S3Uri"].rstrip("/") == offline_uri.rstrip("/"), "Existing offline location differs."
        if kms_key:
            assert actual_storage.get("KmsKeyId") == kms_key, "Existing encryption key differs."
        catalog = description["OfflineStoreConfig"]["DataCatalogConfig"]
        group.update(database=catalog["Database"], table=catalog["TableName"], catalog=catalog["Catalog"])
        print("Ready:", group["name"], "| Table:", group["database"] + "." + group["table"])
    (RUN / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
    print("Saved settings:", (RUN / "settings.json").resolve())
else:
    print("Skipped: CSV mode.")

## Ingest the generated rows

- Skip this cell entirely in CSV mode.
- Send generated records to the groups and retain a delivery receipt.
- Ingestion happens here, separately from each model pipeline execution.

In [ ]:
if data_source == "feature_store":
    receipt = ingest_csv(
        session, settings, sample["sample"], delivery_id=RUN.name,
        upstream_revision="metadata-sha256:" + settings["source_metadata"]["metadata_sha256"],
    )
    (RUN / "ingestion.json").write_text(json.dumps(receipt, indent=2) + "\n")
    print("Completed delivery:", receipt["delivery_id"])
    print("Records per group:", receipt["records"])
    print("Receipt:", (RUN / "ingestion.json").resolve())
else:
    print("Skipped: CSV mode.")

## Data inputs for notebook 02

- Copy the complete two-line input block printed by this cell into notebook 02's data-input cell.
- CSV mode supplies the generated sample.csv path; Feature Store mode supplies the binding URI.
- Set the same data_source in notebook 02's separate Defaults cell; there is no permission-error fallback.

In [ ]:
if data_source == "feature_store":
    from mlops_features.data import write_once
    config_key = settings["prefix"] + "configurations/" + uuid4().hex + "/feature-store.json"
    write_once(session.client("s3"), bucket, config_key, settings, kms_key=kms_key)
    feature_store_config_uri = f"s3://{bucket}/{config_key}"
    print("Use data_source = 'feature_store' in notebook 02's Defaults cell.")
    print("Completed delivery ID:", receipt["delivery_id"])
    print("Leave FeatureStoreDeliveryId blank for the most recently completed delivery, or set this ID to replay it.")
    print("\nCopy this complete block into notebook 02's data-input cell:")
    print('csv_input = ""')
    print("feature_store_config_uri =", repr(feature_store_config_uri))
else:
    feature_store_config_uri = ""
    print("Use data_source = 'csv_direct' in notebook 02's Defaults cell.")
    print("Feature Store was not called. Notebook 02 uploads the generated sample.csv for GetData.")
    print("\nCopy this complete block into notebook 02's data-input cell:")
    print("csv_input =", repr(str(sample["sample"].resolve())))
    print('feature_store_config_uri = ""')

## Optional offline readback

- Run only when Feature Store mode and verify_offline_readback are both enabled.
- Query the delivered rows with Athena and compare them with the generated sample.
- Write local readback evidence; this can wait for offline-store visibility.

In [ ]:
if data_source == "feature_store" and verify_offline_readback:
    DATASET = RUN / ("readback-" + uuid4().hex)
    dataset = read_dataset(session, settings, receipt, DATASET, publish=False)
    types = {"String": str, "Integral": "Int64", "Fractional": float}
    csv_types = {name: types[kind] for group in groups.values() for name, kind in group["columns"].items()}
    original = pd.read_csv(sample["sample"], dtype=csv_types, keep_default_na=False, na_values=[""])
    retrieved = pd.read_csv(DATASET / "data.csv", dtype=csv_types, keep_default_na=False, na_values=[""])
    original[event_time] = pd.to_datetime(original[event_time], utc=True).map(lambda value: value.isoformat().replace("+00:00", "Z"))
    keys = [record_id, event_time]
    pd.testing.assert_frame_equal(
        original.sort_values(keys).reset_index(drop=True),
        retrieved.sort_values(keys).reset_index(drop=True),
        check_exact=False, rtol=1e-12, atol=1e-12,
    )
    print("Verified offline values:", len(retrieved), "rows")
    print("Local diagnostic files:", DATASET.resolve())
    display(retrieved.iloc[:5, :10])
else:
    print("Skipped: optional readback is disabled.")